# K4-1000 temporal context 비교 · LIBERO+ Long

같은 K4-1000과 이전 평가의 `libero_10` 태스크 15개를 사용합니다. 두 조건에서 태스크당 1회, 총 30회 rollout을 실행합니다.

| 조건 | context 초기화 시점 |
|---|---|
| `episode_only` | 매 에피소드 첫 번째 정책 쿼리 |
| `every_4_queries` | 매 에피소드의 1, 5, 9, … 번째 정책 쿼리 |

두 조건에서 seed 42, 기본 초기 상태 0번, 8-step open-loop와 기존 action-feature 설정을 사용합니다. 에피소드마다 실제 초기 상태의 해시, 정책 쿼리 수, context 초기화 위치를 기록합니다. 두 조건의 초기 상태 해시가 일치해야 결과를 집계합니다.

Colab에서 GPU 런타임을 선택한 뒤 순서대로 실행하세요. 앞선 평가 환경이 같은 VM에 있으면 재사용합니다. 마지막 셀이 rollout을 시작하며, Drive의 `VLADE/evaluations/libero_plus_ava_temporal_context` 아래 새 폴더에 결과·영상·로그를 저장합니다.

[이전 Long 실행 설정](https://drive.google.com/file/d/1_d2LObAy5s0pPZ-aBY4HDtXolEFPwnJf/view) · [기존 15개 태스크](https://drive.google.com/file/d/1NvngQde123XWaPwYVTRmVgMfUC2kx4su/view)


In [ ]:
# @title 1. Drive 연결 및 기존 Long 평가 설정
from pathlib import Path
from google.colab import drive
import datetime
import hashlib
import json
import os
import shutil
import subprocess
import tempfile
import zipfile

drive.mount('/content/drive')
DRIVE = Path('/content/drive/MyDrive')
PROJECT = Path('/content/drive/.shortcut-targets-by-id/1jbXOZooUbe-SM9oUrKmNac_4Dfthkgs0/VLADE')
CODE = Path('/content/vlade_libero_plus_eval')
AVA = CODE / 'third_party/ava_vla'
LIBERO = CODE / 'LIBERO-plus'
SCRIPT = AVA / 'experiments/robot/libero/run_libero_eval.py'
VENV = CODE / 'py311-mujoco3.3.1'
PYTHON = VENV / 'bin/python'
ASSETS_ROOT = LIBERO / 'libero/libero'
CHECKPOINT = Path('/content/drive/MyDrive/VLADE/runs/ava_fft_k4/fish13031e-ava-suite-k4-seed42-20261111T062518Z-2f8a601c/main_fft/checkpoints/step-0001000')
LIBERO_REV = '4976dc30028e805ff8094b55501d532c48fec182'
ASSETS_REV = 'dd2bd61b7d9a6fef1abc52d606e983b41886a149'
SUITE = 'libero_10'
TASK_IDS = [102, 122, 130, 356, 383, 419, 456, 571, 895, 914, 1003, 1126, 1728, 2233, 2418]
CONDITIONS = {'episode_only': 0, 'every_4_queries': 4}
BASE_ARGS = [
    "--model_family",
    "openvla",
    "--use_l1_regression",
    "True",
    "--use_diffusion",
    "False",
    "--use_proprio",
    "True",
    "--num_images_in_input",
    "2",
    "--num_open_loop_steps",
    "8",
    "--task_suite_name",
    "libero_10",
    "--num_trials_per_task",
    "1",
    "--seed",
    "42",
    "--use_wandb",
    "False",
    "--multi_frame.temporal_strategy",
    "attn_weight",
    "--multi_frame.temporal_feature_source",
    "action",
    "--multi_frame.temporal_feature_layer",
    "-2",
    "--multi_frame.attn_weight_force_eager_attn",
    "True",
    "--multi_frame.attn_weight_extra_layer_ids",
    "(i for i in range(0, 32))",
    "--multi_frame.attn_weight_head_type",
    "softmaxscore",
    "--multi_frame.attn_weight_score_config",
    "(1.9, 0.1, 0.0)",
    "--multi_frame.attn_weight_sink_ids",
    "[68,75,180,187,324,331,436,443]",
    "--multi_frame.attn_weight_sink_weight",
    "1.0"
]
FORCE_SETUP = False

for name in ('COMPLETE', 'config.json', 'dataset_statistics.json'):
    if not (CHECKPOINT / name).is_file():
        raise FileNotFoundError(CHECKPOINT / name)
print('K4-1000:', CHECKPOINT)
print('Task IDs:', TASK_IDS)


In [ ]:
# @title 2. 기존 평가 환경 재사용 또는 설치
def cached_eval_ready():
    receipt = ASSETS_ROOT / '.vlade_assets_revision'
    if not (PYTHON.is_file() and SCRIPT.is_file() and receipt.is_file()
            and (ASSETS_ROOT / 'assets' / 'textures').is_dir()):
        return False
    if receipt.read_text().strip() != ASSETS_REV:
        return False
    revision = subprocess.run(
        ['git', '-C', str(LIBERO), 'rev-parse', 'HEAD'],
        capture_output=True, text=True,
    )
    return revision.returncode == 0 and revision.stdout.strip() == LIBERO_REV

if not FORCE_SETUP and cached_eval_ready():
    print('기존 LIBERO+ 평가 환경을 재사용합니다.')
else:
    print('기존 평가 노트북의 환경 설치를 실행합니다.')
    AVA_INVENTORY = {'third_party/ava_vla/.pre-commit-config.yaml': '3f5526e4c92fcd179b8c7df3125d07272a9b4fe779b486052026aaa2daaa8f6b', 'third_party/ava_vla/LICENSE': '776b1de6987e63f473063601e3bf65633e81b4ed5ba2b5eebac6b5c5499c2e98', 'third_party/ava_vla/README.md': '3444db84c35a2882c5ea2f2ceb01f855de827416723a6a073b3766347bb893ee', 'third_party/ava_vla/experiments/robot/aloha/aloha_utils.py': 'e91291733b35f33539155989e087cbd24ed0aa1d81acf61cd65eb3d2f6dcb283', 'third_party/ava_vla/experiments/robot/aloha/constants.py': '3498c193544158b7b1156524a51af4886fc57d283d78ad54c6b2921d54779aa6', 'third_party/ava_vla/experiments/robot/aloha/preprocess_split_aloha_data.py': 'bb69c8a0453f0aef5969cfaca9bce11d4e8c5ef1fa5c3179ad55bc8dbf93058a', 'third_party/ava_vla/experiments/robot/aloha/real_env.py': '30006c4152bd0e22e982d4b4acda6062a71d59995e507915da90778457aa4631', 'third_party/ava_vla/experiments/robot/aloha/requirements_aloha.txt': '560784c894b0db4aed26299c436a8f31f70bb41c01646516ce5ef1c93df63f9c', 'third_party/ava_vla/experiments/robot/aloha/robot_utils.py': 'dc9a21730db88dba4ac922fc1e3700eb18bb8392c2f8a9d987b2fe5e3d1e4a72', 'third_party/ava_vla/experiments/robot/aloha/run_aloha_eval.py': 'c3e1929a311c603f781356ca13ff8c18ba9b300643b7097357f9431716d10e90', 'third_party/ava_vla/experiments/robot/calvin/__init__.py': '9b5448c1c0153d89e827224f4987f27a1f4609af43b5198e13946f705d9c4499', 'third_party/ava_vla/experiments/robot/calvin/calvin_env_wrapper.py': '4ff951474d5d0ef3bb1df7af14fc4147864a1733d441f3729ecf7896e4a290b2', 'third_party/ava_vla/experiments/robot/calvin/run_calvin_eval.py': 'de696ff04121e1b9f792526f2c3f18be598591381416a7580ca05c4cd5d15afa', 'third_party/ava_vla/experiments/robot/calvin/vla_evaluation.py': '8994476efecd971ffc40d63cbbe2b76cc59c9b709ad49a39c086d266946308de', 'third_party/ava_vla/experiments/robot/libero/libero_requirements.txt': '209837422576c29c62067b50cb6be9d0bb2b3ef6bd193f544612a192e477cbd3', 'third_party/ava_vla/experiments/robot/libero/libero_utils.py': '287d7d3663fa8f40d14ee9282e13e4a31065d250f436174dbab69d8288e8d708', 'third_party/ava_vla/experiments/robot/libero/regenerate_libero_dataset.py': '165fd2c08a6d29306de53b3ea9a0fbaf4a7e3d533d82464f21a38f2909ac0dd8', 'third_party/ava_vla/experiments/robot/libero/run_libero_eval.py': '3f2c36f5bda7035b7406510c1394e2207afeccb9d5829ddc04c88ff55d31e21f', 'third_party/ava_vla/experiments/robot/openvla_utils.py': 'c852bfd1ce8eddda21dc0c5b07a81ae8af7fa3325645ed52fc256d4f7a93789a', 'third_party/ava_vla/experiments/robot/robot_utils.py': '006234f1c097aeef51e553c865ce0c9d33d89f423c631ed51e3b26aee481f5f8', 'third_party/ava_vla/prismatic/__init__.py': '244d4c09ae7b191ae233484aed3a3dfb281bd1eb7af4e24392f047d0d42c6ef9', 'third_party/ava_vla/prismatic/conf/__init__.py': '2ce220a223ac5b351b6b9b3cd9ad8b16e880394e1b29dcd4b56753a7ddb72cbb', 'third_party/ava_vla/prismatic/conf/datasets.py': '0246b302e816c545738970cee2e881bcc6f7897593cce27c23d2b0c0baa4b1fa', 'third_party/ava_vla/prismatic/conf/models.py': '1f59b22a961c22ef0f1b514a4307d31707516f0617889aebefe852c6c0de4421', 'third_party/ava_vla/prismatic/conf/vla.py': '749a47163db615312b4daf847b987d157bc93cd62279e8fc83dff5a521b71069', 'third_party/ava_vla/prismatic/extern/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'third_party/ava_vla/prismatic/extern/hf/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'third_party/ava_vla/prismatic/extern/hf/configuration_prismatic.py': 'e13181aec9e6a789f876f51b9edbb65c58fccb0e85acb47304c7824938abfd4c', 'third_party/ava_vla/prismatic/extern/hf/modeling_prismatic.py': '9c1a50fa37a1e6843ae0eb1be016acc5d61e8d83b06b60b9ef0ce74ebc44ecb7', 'third_party/ava_vla/prismatic/extern/hf/processing_prismatic.py': '7c091d514ecde538fcc3491717dfe1e16296032530a2664bf472761a0cd9c579', 'third_party/ava_vla/prismatic/models/__init__.py': '5916dda5c2320fbab54f3ba09d67c8a52b2a5be7909d4c652caf0899d9f5eedb', 'third_party/ava_vla/prismatic/models/action_heads.py': 'c011bf5b2e0b05772c90081047122e50acc259923328810ff6c2fd148160034c', 'third_party/ava_vla/prismatic/models/attn_weight_generator.py': '939f1364618e47244c48fd032dbb645e7fc4aaa02a0f0a28e017a884e6b48c62', 'third_party/ava_vla/prismatic/models/backbones/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'third_party/ava_vla/prismatic/models/backbones/llm/__init__.py': '02791236db3febb7ff314310cd6dd8bac174bb7961118a3403995c60d8a95c09', 'third_party/ava_vla/prismatic/models/backbones/llm/base_llm.py': 'ef5b6e0d3d25e1ff177c82b58feae25ccd616f71c104d5fddf491fd4975bca6b', 'third_party/ava_vla/prismatic/models/backbones/llm/llama2.py': '8414694c9099da13831c0540dbc9bd851d172a28340ec4e39d18e2db264d7ba6', 'third_party/ava_vla/prismatic/models/backbones/llm/mistral.py': '4fc1aaaa966b9483bc1cbe98a0a98b4cd78b31b229c04fc97fdff1b082c03ac2', 'third_party/ava_vla/prismatic/models/backbones/llm/phi.py': 'a6d77b7e40b9175b0262910efcfa69347c11a9c94ab7207e523ce33afd90f330', 'third_party/ava_vla/prismatic/models/backbones/llm/prompting/__init__.py': '6f532ea1f274096e32a5b4bf9b98648f3e0521f1f7e30599487a3230663d7b92', 'third_party/ava_vla/prismatic/models/backbones/llm/prompting/base_prompter.py': '13640ed54b055c0fcd8daf7212be9f4faf76cec779fdd2e0f7ea5a866b0b6d51', 'third_party/ava_vla/prismatic/models/backbones/llm/prompting/llama2_chat_prompter.py': 'a35a82facfca67e202ad4ea3aaa778c72776b59eb8ee8936af8ce300ad49f2df', 'third_party/ava_vla/prismatic/models/backbones/llm/prompting/mistral_instruct_prompter.py': '3d5f57db4028ef84f8d09fb83e07f2049985c90a74cec471c764e353fe0fc617', 'third_party/ava_vla/prismatic/models/backbones/llm/prompting/phi_prompter.py': '7828ed56641f2251a5b0b45a2a6432834a24897854bed83434643a7008fc3b30', 'third_party/ava_vla/prismatic/models/backbones/llm/prompting/vicuna_v15_prompter.py': '06e1c31246300773973af7e5f0a36a2b2452653b81be1e983fbfaab5b506ae18', 'third_party/ava_vla/prismatic/models/backbones/vision/__init__.py': 'ed36c168ff2c4e230f7b31b3b68484aa48fb6700b58f14337e327126a00da2eb', 'third_party/ava_vla/prismatic/models/backbones/vision/base_vision.py': '4650adf0c2abe7bb23bb0010a0d00f3ffdc3585e298cba69a36c77cb3af97f5b', 'third_party/ava_vla/prismatic/models/backbones/vision/clip_vit.py': '2529c7a514aabe1d864654cb9783626e9252364cfb939c08f97c0db99f2021aa', 'third_party/ava_vla/prismatic/models/backbones/vision/dinoclip_vit.py': 'f4804c952dd7eb73bfad729934b11442f6f0294ddc1e1e37fa1e0b5d2bc293a0', 'third_party/ava_vla/prismatic/models/backbones/vision/dinosiglip_vit.py': '3799119de6d5d4f471eb841b30135e78efd8a8199b3c4af3db78f1a3a99fcfe3', 'third_party/ava_vla/prismatic/models/backbones/vision/dinov2_vit.py': 'd55c2d1639f4800f34618daecc7b84b794f72e55ba1579cb8385f5d914512a91', 'third_party/ava_vla/prismatic/models/backbones/vision/in1k_vit.py': '8644aabfab30e9fc3a8296d04ff69f6f2b9bd2ebb20143fb5c61e15b64e56fc9', 'third_party/ava_vla/prismatic/models/backbones/vision/siglip_vit.py': '7098e14baf02cbc0e814dc23a5abe31948529556fd7f5cd156ba73fdcce381b3', 'third_party/ava_vla/prismatic/models/film_vit_wrapper.py': '36d90b2717f210b9a0da8f80ba0d33edb0916ef75745caa20d8f6f266a92da3f', 'third_party/ava_vla/prismatic/models/load.py': '2f9cc4535e5fc190669a69463b55cdc35709ce6f990055ecd9e20a0997e88604', 'third_party/ava_vla/prismatic/models/materialize.py': 'cd7aecf09de31b7fb18ac2795bf7b5f97f9e07157dbfd5d588e6414d9904d94a', 'third_party/ava_vla/prismatic/models/projectors.py': 'ca0fc18af5ba38501a9642ac6da4a64047e2ed013cd976ea3da2632ca33eb2e1', 'third_party/ava_vla/prismatic/models/registry.py': '4e226a6f6182ec4ecbdfd8507c85831cbc43de03a5cbb715af712092a7482b21', 'third_party/ava_vla/prismatic/models/vlas/__init__.py': 'b55e834c024738ce1ca7f69392c01262f19078343c57f264177fe347a694dfe0', 'third_party/ava_vla/prismatic/models/vlas/openvla.py': 'c364351dc787ad7a6fb5ee6168532a7c4ba6d0a2c98b65836f20708524d71337', 'third_party/ava_vla/prismatic/models/vlms/__init__.py': '0f5b26ba8114b3f1717aa693568bfb3dcf52ab7b2f2b6d7fe0bd69ff19751817', 'third_party/ava_vla/prismatic/models/vlms/base_vlm.py': '1c0c8815d745bc01f8c6d93b893aded94cdc55fe6ffcbdf12a255a8784b3921f', 'third_party/ava_vla/prismatic/models/vlms/prismatic.py': '612b4369e77880c3fd3ae7960fdf136b745d6a2dce57176d12160f30d88de20f', 'third_party/ava_vla/prismatic/overwatch/__init__.py': '3e9c13fa52c707586621c75406253086340c4667904d0053e3f34ee991b2241a', 'third_party/ava_vla/prismatic/overwatch/overwatch.py': '7e065cd22a6337a1b9a43fb4f456d7fed1231345b147f7a2a0f48cfc118af45d', 'third_party/ava_vla/prismatic/preprocessing/__init__.py': '5681e8f6c09e4d86198b8107b9d7251c90b5b455b04080827d28f37d8bf56372', 'third_party/ava_vla/prismatic/preprocessing/datasets/__init__.py': '72e19b90d709ebd6178bf879e5f9dfecbc83631642c8d31c264c955733cb4ef1', 'third_party/ava_vla/prismatic/preprocessing/datasets/datasets.py': 'e51471c98f85fba6f4e2fe03204bbbf2a176163931e700dde2310affb5832575', 'third_party/ava_vla/prismatic/preprocessing/download.py': '257fda53e11217f3c3cd0b9092c8064fe8a7af5910edf9bd0cb8f7f1d29c1b9d', 'third_party/ava_vla/prismatic/preprocessing/materialize.py': 'f98ab5e3ba03b6cbe348e8a488a41ece3633cb7c46fd82836a20f10484938772', 'third_party/ava_vla/prismatic/training/__init__.py': '625226108c7459cc0fed8cb26193496980648c1060d2ff42a8a9400dd3bf51d6', 'third_party/ava_vla/prismatic/training/materialize.py': '80d693f2144989ac5687fb243b4ca9a6648e25df10c83cf493f0010b5f657b12', 'third_party/ava_vla/prismatic/training/metrics.py': 'd0ee5dee7f7141a4be5b4622580abe6f1b6d183cb995abcbcf1cbb120d1a6fc2', 'third_party/ava_vla/prismatic/training/strategies/__init__.py': '7d65535181ee4fd105dae95e69e5de1a7599a9f31bb8f97314149c62807ad06d', 'third_party/ava_vla/prismatic/training/strategies/base_strategy.py': '5fb67f5d867af89e559c1f3e64292c8018285c10950d8b6baf078cea5f728158', 'third_party/ava_vla/prismatic/training/strategies/ddp.py': 'db038a951ad1d8d0ffd70f8e4d8b58d89c85e209f7a8aaf1ad4eb408266d0138', 'third_party/ava_vla/prismatic/training/strategies/fsdp.py': 'ca1b90934237e604ef02f017f79ea7eed3d5d9c1431e9512f92a2493e4fcc52d', 'third_party/ava_vla/prismatic/training/train_utils.py': '6ecc95436fde63ce023e2ba7151826d96a915b5d5c2ec5698750022580b0e7b3', 'third_party/ava_vla/prismatic/util/__init__.py': '8ab7190ed1f1bd087966fcc59f35f9a7b4b315dd7b17f68a6a7ed6e335fc7380', 'third_party/ava_vla/prismatic/util/batching_utils.py': 'a41d58f7baa40ccefe92be2727c19963a720a41feeada301a09cf01c8b261feb', 'third_party/ava_vla/prismatic/util/data_utils.py': '431ef4210296c2165142b1226a4adad985d99d0391cd822f1630e98fbb6c9d7d', 'third_party/ava_vla/prismatic/util/nn_utils.py': '7aa0562eab374f0585bb53639b861f06b6c44b82384e938dda9cd32d844bc0d0', 'third_party/ava_vla/prismatic/util/temporal_context_utils.py': '169a74905a36015f00a921558cf76e27cc18f502daf20beddab32a5c55032da2', 'third_party/ava_vla/prismatic/util/torch_utils.py': 'a70e3b8d7e24b15a5d8abfa94f1c596ea70a9a85c86b50e8d8f963af51ff3304', 'third_party/ava_vla/prismatic/vla/__init__.py': 'bfc31eb7e16374695cc0da879bb1e43451395ee64144da896f00e2187862f069', 'third_party/ava_vla/prismatic/vla/action_tokenizer.py': 'fdc98fcbf5b0926ef2181db71946d23ffbfa052cf8443dc933d52c42a191352c', 'third_party/ava_vla/prismatic/vla/constants.py': '5b53c3a07cf1ed27bf94a88da9929e82818792d6fdc32343d133afea1a0acf3d', 'third_party/ava_vla/prismatic/vla/datasets/__init__.py': 'f40c0fa32bbb2f52d9a8a4cd1acfa061b4e0da0e0e6e4dc290ae6f351cf62f04', 'third_party/ava_vla/prismatic/vla/datasets/datasets.py': '36254f1112e66af6c1394ebf8279fc505bb327306ce892c78f321831b34ca739', 'third_party/ava_vla/prismatic/vla/datasets/rlds/__init__.py': '35b3206143eaf348bdb05401dddc4c14b382761be2832d8f64a8e6bf8e4c404c', 'third_party/ava_vla/prismatic/vla/datasets/rlds/dataset.py': '4acfd0ebfe7e2765de1a4e5fd19ceb0939cd77ef1b022756f47871ccdae260f1', 'third_party/ava_vla/prismatic/vla/datasets/rlds/obs_transforms.py': '1f2eddd9e2fe5ea96be294945567e665172287973be5753a7381d777c7d40392', 'third_party/ava_vla/prismatic/vla/datasets/rlds/oxe/__init__.py': '4a8a96f8c2af6c5b56f2bc29493141c160dcd3595ca977a95245b2ba64598157', 'third_party/ava_vla/prismatic/vla/datasets/rlds/oxe/configs.py': '0995257a400c5d396cb75123cd3567283d701ba89a037dc7e0aecba7e2dcbf9b', 'third_party/ava_vla/prismatic/vla/datasets/rlds/oxe/materialize.py': '375c8117e02fec5ff5971d1d9c1fb021ec2e615aa357be21bbb3d5e5d5001496', 'third_party/ava_vla/prismatic/vla/datasets/rlds/oxe/mixtures.py': '0598aac5251f44d70d5e0c609977d32afcda30c72098329005d8559bb35786bd', 'third_party/ava_vla/prismatic/vla/datasets/rlds/oxe/transforms.py': '7644f1b4fa0badab70f9924de68eeb9d94b41c82bf6d8dbc776e58677adae485', 'third_party/ava_vla/prismatic/vla/datasets/rlds/oxe/utils/droid_utils.py': '6eddeaf36eb335656435f8773ac715ab966994c2b97cf72f257751140c3c4099', 'third_party/ava_vla/prismatic/vla/datasets/rlds/traj_transforms.py': '1ade02497222b16f74da1a005a2d22829d109e3c1855c14b078dc5cbdb799566', 'third_party/ava_vla/prismatic/vla/datasets/rlds/utils/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'third_party/ava_vla/prismatic/vla/datasets/rlds/utils/data_utils.py': '4f7a36c6713e6c28a7237ec7ae1325abd47a1f0c921164fc5548ce4c81340d44', 'third_party/ava_vla/prismatic/vla/datasets/rlds/utils/goal_relabeling.py': 'e59594065d9497ca4a3e51fe53d6687fbd6990a966d67a36528524e7cf22d360', 'third_party/ava_vla/prismatic/vla/datasets/rlds/utils/task_augmentation.py': '1593b89224e6d70552b13f182e42194ea348b5dd1ea9b0507b4fa09642f25113', 'third_party/ava_vla/prismatic/vla/materialize.py': 'dbc1b0c88e3acb926829e7d86f00b7aef52b6679af641f5ad328aa59d99f7ab5', 'third_party/ava_vla/pyproject.toml': '906f17ee278cef3bb4352f0dc80967986d8686151d3905856fa69f673bbb9f28', 'third_party/ava_vla/scripts/extern/convert_prismatic_weights_to_hf.py': '75d9921fa41d4c3a5540aa478d8057da47fb4dc324c74737557cb0abd76f23a2', 'third_party/ava_vla/scripts/extern/verify_prismatic.py': 'a0d13b4960997f8d09667139ef170c3dd9257871b44e04cc0d3437dba55bb783', 'third_party/ava_vla/vla-scripts/deploy.py': '4d00d69b594d694181e05d6da7e675b5a1d57c72c310a7e8706700c49f16aca5', 'third_party/ava_vla/vla-scripts/extern/convert_openvla_weights_to_hf.py': 'f1ad6f8db1e5c52a5fc39d8f4f487c62cd6308e2469805358efbb02f4c505809', 'third_party/ava_vla/vla-scripts/extern/verify_openvla.py': 'baee65706be73d70a8b163b223ae9f4a91141c21edafd5f7ee0976b3ff7ad47f', 'third_party/ava_vla/vla-scripts/finetune.py': '1400729cd444559dd914f9c3551868708592c79cbab96462cf80ad8c2bb0699c', 'third_party/ava_vla/vla-scripts/merge_lora_weights_and_save.py': '84bd8d83a511e7e98e9160c1b69d3508ac8e40586f98cc3b9f6b9c781733a7f1'}
    for rel, expected in AVA_INVENTORY.items():
        dst = CODE / rel
        if dst.is_file() and hashlib.sha256(dst.read_bytes()).hexdigest() == expected:
            continue
        payload = (PROJECT / rel).read_bytes()
        if hashlib.sha256(payload).hexdigest() != expected:
            raise ValueError(f'AVA source SHA mismatch: {rel}')
        dst.parent.mkdir(parents=True, exist_ok=True)
        dst.write_bytes(payload)
    import shutil, subprocess, os

    def run_cmd(cmd, cwd=None, env=None):
        print('RUN:', ' '.join(map(str,cmd)), flush=True)
        subprocess.run(list(map(str, cmd)), cwd=str(cwd) if cwd else None, env=env, check=True)

    run_cmd(['apt-get','update','-qq'])
    run_cmd(['apt-get','install','-y','python3.11','python3.11-venv','python3.11-dev',
             'build-essential','linux-libc-dev'])
    if not PYTHON.is_file():
        run_cmd(['python3.11','-m','venv',str(VENV)])
    run_cmd([PYTHON,'-m','pip','install','-U','pip','setuptools','wheel'])
    # AVA 원본 pyproject의 torch2.2/cu121 (기존 평가 노트북과 동일)
    run_cmd([PYTHON,'-m','pip','install','torch==2.2.0','torchvision==0.17.0','torchaudio==2.2.0',
             '--index-url','https://download.pytorch.org/whl/cu121'])
    run_cmd([PYTHON,'-m','pip','install','-e',str(AVA)])
    # LIBERO: 설치 이전에 Linux 빌드 패키지를 준비해 evdev wheel 오류 차단
    run_cmd([PYTHON,'-m','pip','install','evdev'])
    run_cmd([PYTHON,'-m','pip','install',
             'robosuite==1.4.1','bddl','future','easydict==1.9','gym==0.25.2',
             'termcolor','h5py','tqdm','PyYAML','imageio','imageio-ffmpeg',
             'numpy==1.26.4','opencv-python<4.12',
             'tensorflow-metadata==1.15.0'])
    # 이전 rollout 노트북 18번 셀의 명시 버전 (2.3.7 vs 3.3.1 혼합 금지)
    run_cmd([PYTHON,'-m','pip','install','--no-deps','mujoco==3.3.1'])

    if not LIBERO.exists():
        run_cmd(['git', 'clone', 'https://github.com/sylvestf/LIBERO-plus.git', LIBERO])
    run_cmd(['git', '-C', LIBERO, 'checkout', '--detach', LIBERO_REV])
    actual_revision = subprocess.check_output(['git', '-C', str(LIBERO), 'rev-parse', 'HEAD'], text=True).strip()
    assert actual_revision == LIBERO_REV
    run_cmd([PYTHON, '-m', 'pip', 'install', '--no-deps', '-e', LIBERO])
    assets_root = LIBERO / 'libero/libero'
    receipt = assets_root / '.vlade_assets_revision'
    if not receipt.is_file() or receipt.read_text().strip() != ASSETS_REV:
        archive = PROJECT / 'data/raw/libero_plus_assets/assets.zip'
        if not archive.is_file():
            archive = Path(subprocess.check_output([
                str(PYTHON), '-c',
                'from huggingface_hub import hf_hub_download; import sys; print(hf_hub_download("Sylvest/LIBERO-plus","assets.zip",repo_type="dataset",revision=sys.argv[1]))',
                ASSETS_REV], text=True).strip())
        with zipfile.ZipFile(archive) as files:
            for name in files.namelist():
                relative = Path(name)
                if relative.is_absolute() or '..' in relative.parts:
                    raise ValueError(f'Invalid assets archive member: {name}')
            files.extractall(assets_root)
        if not (assets_root / 'assets' / 'textures').is_dir():
            raise RuntimeError('LIBERO+ asset extraction incomplete')
        receipt.write_text(ASSETS_REV)

In [ ]:
# @title 3. GPU·시뮬레이터·MP4 검사 및 체크포인트 준비
ENV = {
    **os.environ,
    'MPLBACKEND': 'Agg',
    'MUJOCO_GL': 'egl',
    'PYOPENGL_PLATFORM': 'egl',
    'PYTHONPATH': f'{AVA}:{CODE / "src"}:{LIBERO}:' + os.environ.get('PYTHONPATH', ''),
    'TOKENIZERS_PARALLELISM': 'false',
    'WANDB_MODE': 'disabled',
}
subprocess.run(
    [str(PYTHON), '-m', 'pip', 'install', 'imageio-ffmpeg'],
    env=ENV, check=True,
)
PREFLIGHT = r'''
import sys
import tempfile
from pathlib import Path

import imageio.v2 as imageio
import mujoco
import numpy as np
import torch
from libero.libero import benchmark
from libero.libero.envs import OffScreenRenderEnv

assert torch.cuda.is_available(), 'Colab GPU를 연결하세요.'
assert mujoco.__version__ == '3.3.1', mujoco.__version__
suite = benchmark.get_benchmark_dict()['libero_10']()
assert suite.n_tasks == 2519
env = OffScreenRenderEnv(
    bddl_file_name=suite.get_task_bddl_file_path(int(sys.argv[1])),
    camera_heights=128, camera_widths=128,
)
try:
    obs = env.reset()
    assert isinstance(obs, dict) and obs
finally:
    env.close()
with tempfile.TemporaryDirectory() as temp:
    video = Path(temp) / 'test.mp4'
    with imageio.get_writer(str(video), fps=30) as writer:
        writer.append_data(np.zeros((32, 32, 3), dtype=np.uint8))
    assert video.stat().st_size > 0
print('PREFLIGHT OK', torch.cuda.get_device_name(0))
'''
subprocess.run(
    [str(PYTHON), '-c', PREFLIGHT, str(TASK_IDS[0])],
    cwd=LIBERO, env=ENV, input='N\n', text=True, timeout=300, check=True,
)

view_root = Path(tempfile.mkdtemp(prefix='ava_temporal_context_', dir='/content'))
CHECKPOINT_VIEW = view_root / 'K4-1000'
CHECKPOINT_VIEW.mkdir()
for source in CHECKPOINT.iterdir():
    if not source.is_file() or source.name in {'trainer_state.pt', 'optimizer.pt'}:
        continue
    target = CHECKPOINT_VIEW / source.name
    if source.suffix in {'.json', '.py'}:
        shutil.copy2(source, target)
    else:
        target.symlink_to(source)

stats_path = CHECKPOINT_VIEW / 'dataset_statistics.json'
config_path = CHECKPOINT_VIEW / 'config.json'
stats = json.loads(stats_path.read_text())
config = json.loads(config_path.read_text())
stats[SUITE] = stats['libero_plus_mixdata']
config.setdefault('norm_stats', {}).update({
    'libero_plus_mixdata': stats['libero_plus_mixdata'],
    SUITE: stats[SUITE],
})
stats_path.write_text(json.dumps(stats, indent=2))
config_path.write_text(json.dumps(config, indent=2))
print('K4-1000 평가 준비 완료:', CHECKPOINT_VIEW)


In [ ]:
# @title 4. Long paired rollout 실행 및 집계
EVAL_RUNNER = "import hashlib\nimport importlib.util\nimport inspect\nimport json\nimport sys\nfrom pathlib import Path\n\nimport numpy as np\n\nscript, result_file, suite_name, reset_period, task_json = sys.argv[1:6]\nsys.argv = [script, *sys.argv[6:]]\nsys.path.insert(0, str(Path(script).parent))\nreset_period = int(reset_period)\ntask_ids = json.loads(task_json)\ntask_set = set(task_ids)\nresult_path = Path(result_file)\n\nspec = importlib.util.spec_from_file_location('vlade_context_eval', script)\nmodule = importlib.util.module_from_spec(spec)\nsys.modules[spec.name] = module\nspec.loader.exec_module(module)\nsuite = module.benchmark.get_benchmark_dict()[suite_name]()\nif not task_set <= set(range(suite.n_tasks)):\n    raise ValueError('Task ID is outside the pinned benchmark.')\n\nrun_task = module.run_task\nrun_episode = module.run_episode\nget_action = module.get_action\nlog_message = module.log_message\ntask_signature = inspect.signature(run_task)\nepisode_signature = inspect.signature(run_episode)\nrecords = []\nlast_episode = None\n\n\ndef fail_on_episode_error(message, *args, **kwargs):\n    log_message(message, *args, **kwargs)\n    if str(message).startswith('Episode error:'):\n        raise RuntimeError(message)\n\n\ndef traced_episode(*args, **kwargs):\n    global last_episode\n    bound = episode_signature.bind(*args, **kwargs)\n    bound.apply_defaults()\n    initial_state = bound.arguments['initial_state']\n    if initial_state is None:\n        raise RuntimeError('Expected a fixed benchmark initial state.')\n    state = np.ascontiguousarray(initial_state)\n    digest = hashlib.sha256()\n    digest.update(repr((state.shape, state.dtype.str)).encode())\n    digest.update(state.tobytes())\n    query_count = 0\n    reset_queries = []\n\n    def traced_get_action(*action_args, **action_kwargs):\n        nonlocal query_count\n        if 'temporal_context' not in action_kwargs:\n            raise RuntimeError('Evaluator no longer passes temporal_context by keyword.')\n        query_count += 1\n        if query_count == 1 and action_kwargs['temporal_context'] is not None:\n            raise RuntimeError('Episode did not start with an empty context.')\n        if reset_period and (query_count - 1) % reset_period == 0:\n            action_kwargs['temporal_context'] = None\n        if action_kwargs['temporal_context'] is None:\n            reset_queries.append(query_count)\n        return get_action(*action_args, **action_kwargs)\n\n    module.get_action = traced_get_action\n    try:\n        result = run_episode(*args, **kwargs)\n    finally:\n        module.get_action = get_action\n\n    expected = list(range(1, query_count + 1, reset_period)) if reset_period else [1]\n    if not query_count or reset_queries != expected:\n        raise RuntimeError(f'Unexpected context resets: {reset_queries}, expected {expected}')\n    last_episode = {\n        'initial_state_sha256': digest.hexdigest(),\n        'policy_queries': query_count,\n        'context_reset_queries': reset_queries,\n    }\n    return result\n\n\ndef selected_task(*args, **kwargs):\n    global last_episode\n    inputs = task_signature.bind(*args, **kwargs).arguments\n    task_id = int(inputs['task_id'])\n    before = inputs['total_episodes'], inputs['total_successes']\n    if task_id not in task_set:\n        return before\n    last_episode = None\n    after = run_task(*args, **kwargs)\n    success = after[1] - before[1]\n    if after[0] - before[0] != 1 or success not in (0, 1) or last_episode is None:\n        raise RuntimeError(f'Expected one completed rollout for task {task_id}')\n    records.append({'task_id': task_id, 'success': bool(success), **last_episode})\n    temporary = result_path.with_suffix('.tmp')\n    temporary.write_text(json.dumps(records, indent=2))\n    temporary.replace(result_path)\n    return after\n\n\nmodule.log_message = fail_on_episode_error\nmodule.run_episode = traced_episode\nmodule.run_task = selected_task\nmodule.eval_libero()\nif [row['task_id'] for row in records] != task_ids:\n    raise RuntimeError('Incomplete or reordered rollout results.')\n"

run_id = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
output = DRIVE / 'VLADE/evaluations/libero_plus_ava_temporal_context' / run_id
output.mkdir(parents=True, exist_ok=False)
script_sha = hashlib.sha256(SCRIPT.read_bytes()).hexdigest()
print('출력 폴더:', output, flush=True)

by_condition = {}
for name, period in CONDITIONS.items():
    folder = output / name
    folder.mkdir()
    result_path = folder / 'task_results.json'
    args = BASE_ARGS + [
        '--pretrained_checkpoint', str(CHECKPOINT_VIEW),
        '--local_log_dir', str(folder),
    ]
    run_config = {
        'condition': name,
        'reset_every_policy_queries': period or None,
        'checkpoint': str(CHECKPOINT),
        'evaluator_sha256': script_sha,
        'libero_plus_revision': LIBERO_REV,
        'assets_revision': ASSETS_REV,
        'task_ids': TASK_IDS,
        'args': args,
    }
    (folder / 'run.json').write_text(json.dumps(run_config, indent=2))
    command = [
        str(PYTHON), '-u', '-c', EVAL_RUNNER, str(SCRIPT), str(result_path),
        SUITE, str(period), json.dumps(TASK_IDS), *args,
    ]
    log_path = folder / 'console.log'
    print(f'{name} 시작 · 로그 {log_path}', flush=True)
    with log_path.open('w', encoding='utf-8') as log:
        result = subprocess.run(command, cwd=AVA, env=ENV, stdout=log, stderr=subprocess.STDOUT)
    if result.returncode:
        raise RuntimeError(f'{name} 평가 실패. 로그: {log_path}')
    rows = json.loads(result_path.read_text())
    if [row['task_id'] for row in rows] != TASK_IDS:
        raise RuntimeError(f'{name}: 태스크 ID 또는 실행 순서 불일치')
    by_condition[name] = rows
    successes = sum(row['success'] for row in rows)
    print(f'{name} 완료 · {successes}/{len(rows)} ({successes / len(rows):.1%})', flush=True)

pairs = []
for baseline, reset in zip(by_condition['episode_only'], by_condition['every_4_queries'], strict=True):
    task_id = baseline['task_id']
    if baseline['initial_state_sha256'] != reset['initial_state_sha256']:
        raise RuntimeError(f'Task {task_id}: 두 조건의 초기 상태 불일치')
    pairs.append({
        'task_id': task_id,
        'episode_only': baseline['success'],
        'every_4_queries': reset['success'],
        'initial_state_sha256': baseline['initial_state_sha256'],
    })
totals = {name: sum(row['success'] for row in rows) for name, rows in by_condition.items()}
summary = {
    'suite': SUITE,
    'checkpoint': str(CHECKPOINT),
    'task_count': len(TASK_IDS),
    'conditions': {
        name: {'successes': count, 'success_rate': count / len(TASK_IDS)}
        for name, count in totals.items()
    },
    'episode_only_success_tasks': [
        row['task_id'] for row in pairs if row['episode_only'] and not row['every_4_queries']
    ],
    'every_4_queries_success_tasks': [
        row['task_id'] for row in pairs if row['every_4_queries'] and not row['episode_only']
    ],
    'success_rate_delta_percentage_points':
        100 * (totals['every_4_queries'] - totals['episode_only']) / len(TASK_IDS),
    'paired_results': pairs,
}
summary_path = output / 'comparison.json'
summary_path.write_text(json.dumps(summary, indent=2))
print(f"4쿼리마다 초기화 − 에피소드 시작에만 초기화: {summary['success_rate_delta_percentage_points']:+.1f}%p")
print('결과:', summary_path)
